# Simple Calculator MCP Example

This notebook demonstrates how to use the NeMo Agent Toolkit SDK to integrate with Model Context Protocol (MCP) servers. You'll learn to use remote tools through MCP in a hybrid architecture.

## Key Features

1. **Local MCP Server** - Connect to a local MCP server using stdio transport
2. **Remote MCP Server** - Connect to a remote MCP server using streamable-http transport
3. **Tool Overrides** - Customize tool names and descriptions
4. **Hybrid Tool Composition** - Combine multiple MCP sources in a single workflow

## Prerequisites

1. Install dependencies:
   ```bash
   uv pip install mcp-server-time
   ```

2. Start the remote MCP server (NAT calculator served via MCP):
   ```bash
   nat mcp serve --config_file examples/getting_started/simple_calculator/configs/config.yml
   ```

3. Set environment variable:
   - `NVIDIA_API_KEY` - NVIDIA API key


In [ ]:
import os
import sys

# Add src to path for development
module_path = os.path.abspath('../../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)


## Creating the Workflow

We'll create a workflow that combines:
- A local MCP server for time operations (using stdio transport)
- A remote MCP server for calculator operations (using streamable-http transport)


In [ ]:
from pathlib import Path

from pydantic import HttpUrl

from nat.agent.sdk import NatReActAgent
from nat.llm.sdk import NimLLM
from nat.plugins.mcp.client_config import MCPServerConfig
from nat.plugins.mcp.client_config import MCPToolOverrideConfig
from nat.plugins.mcp.sdk import MCPClient
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create the LLM
llm = NimLLM(
    model_name="meta/llama-3.1-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

# Create MCP client for time operations (local stdio transport)
mcp_time = MCPClient(
    server=MCPServerConfig(
        transport="stdio",
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=America/Los_Angeles"],
    ),
    tool_overrides={
        "get_current_time": MCPToolOverrideConfig(
            alias="get_current_time_mcp_tool",
            description="Returns the current date and time",
        ),
    },
    name="mcp_time",
)

# Create MCP client for calculator operations (remote HTTP transport)
# Note: The NAT MCP server must be running on port 9901
mcp_math = MCPClient(
    server=MCPServerConfig(
        transport="streamable-http",
        url=HttpUrl("http://localhost:9901/mcp"),
    ),
    include=[
        "calculator.add",
        "calculator.subtract",
        "calculator.multiply",
        "calculator.divide",
        "calculator.compare",
    ],
    name="mcp_math",
)

# Create the ReAct agent with both MCP clients
agent = NatReActAgent(
    tools=[mcp_time, mcp_math],
    llm=llm,
    verbose=True,
    parse_agent_response_max_retries=3,
)

# Wrap in NatWorkflow
nat_workflow = NatWorkflow(
    entrypoint=agent,
)

print("Workflow created successfully!")


## Saving the Configuration

Save the workflow configuration to a YAML file.


In [ ]:
config_dir = Path(os.getcwd()) / "config"
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "workflow_config.yaml"
nat_workflow.save_to_config_file(config_path)

print(f"Configuration saved to: {config_path}")
print("\n" + "="*50 + "\n")

with open(config_path) as f:
    print(f.read())


## Testing the Workflow

Test the workflow with a query that uses both time and calculator operations.

**Note**: The NAT MCP server must be running for the calculator operations:
```bash
nat mcp serve --config_file examples/getting_started/simple_calculator/configs/config.yml
```


In [ ]:
# Test the workflow (requires NAT MCP server to be running)
# Uncomment to run:
# result = await nat_workflow.prompt(
#     "Is the product of 2 * 4 greater than the current hour of the day?"
# )
# print(result)


In [ ]:
# Additional test queries
# Uncomment to run:

# Basic calculation
# result = await nat_workflow.prompt("What is 25 + 17?")
# print(result)

# Time query
# result = await nat_workflow.prompt("What time is it right now?")
# print(result)

# Combined query
# result = await nat_workflow.prompt("Multiply the current minute by 3")
# print(result)


## Summary

This notebook demonstrated:

1. **MCPClient SDK Class** - Connecting to MCP servers with different transports
2. **Tool Overrides** - Customizing tool names and descriptions
3. **Multi-Transport Architecture** - Combining stdio (local) and streamable-http (remote) transports
4. **Hybrid Tool Composition** - Using multiple MCP servers in a single workflow

### MCP Transport Types

| Transport | Use Case | Example |
|---|---|---|
| `stdio` | Local MCP servers launched as subprocesses | `mcp_server_time` |
| `streamable-http` | Remote MCP servers via HTTP (recommended) | NAT MCP server |
| `sse` | Remote MCP servers via Server-Sent Events (legacy) | Older MCP servers |

### Related Examples

- [Simple Calculator](../../getting_started/simple_calculator/) - The NAT workflow served via MCP
- [Kaggle MCP](../kaggle_mcp/) - MCP client with bearer token authentication
